# Direct Codegen: Exporting PyTorch & LiteRT Models for MATLAB/Simulink

This tutorial walks you through exporting deep learning models from Python so they can be imported into **MATLAB** and **Simulink** for code generation and deployment.

We cover three progressively complex scenarios:

| # | Model | Difficulty | Export Format | Key Concept |
|---|-------|-----------|---------------|-------------|
| 1 | **RepViT** (image classifier) | Simple | PyTorch `.pt2` | One-shot export of a pretrained model |
| 2 | **YOLOv11** (object detector + segmentor) | Medium | LiteRT `.tflite` | Multi-output model export |
| 3 | **Stateful LSTM** (sensor activity recognition) | Advanced | Both `.pt2` and `.tflite` | Stateful model with explicit hidden state I/O |

**How to use this notebook:** Click **Runtime > Run all**, or run each cell one by one. Each section is self-contained.

---

## Step 0: Install Packages

We install PyTorch 2.8.0 (the version supported for MATLAB codegen) and the libraries needed for each model export.

In [ ]:
!pip install -q torch==2.8.0 torchvision --force-reinstall --no-deps
!pip install -q timm litert-torch

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
import timm

# Load pretrained RepViT model
# 'repvit_m1_0' is the base variant; '.dist_450e_in1k' means:
#   - distillation-trained for 450 epochs
#   - on ImageNet-1K (1000 classes)
repvit_model = timm.create_model('repvit_m1_0.dist_450e_in1k', pretrained=True)
repvit_model.eval()

print(f'Model loaded: RepViT M1.0')
print(f'Number of parameters: {sum(p.numel() for p in repvit_model.parameters()):,}')

In [ ]:
# RepViT expects input shape: (batch, channels=3, height=224, width=224)
# - 3 channels for RGB images
# - 224x224 is the standard ImageNet resolution
sample_input = torch.randn(1, 3, 224, 224)

# Quick sanity check: run a forward pass
with torch.no_grad():
    output = repvit_model(sample_input)
print(f'Input shape:  {sample_input.shape}')   # [1, 3, 224, 224]
print(f'Output shape: {output.shape}')          # [1, 1000] — one score per ImageNet class

In [ ]:
# Export the model using torch.export
# This captures the full computation graph for code generation
exported_repvit = torch.export.export(repvit_model, (sample_input,))

# Save as a .pt2 file
torch.export.save(exported_repvit, 'repvit.pt2')
print('Saved: repvit.pt2')
print(f'\nThis file can be loaded in MATLAB using:')
print('  net = loadPyTorchExportedProgram("repvit.pt2")')

That's it! Three lines of code: **load** the model, **export** the graph, **save** the file.

In the MATLAB live demo, we will load this `.pt2` file using `loadPyTorchExportedProgram` and generate C/C++ code from it.

---

# Part 2: YOLOv11 — Object Detection & Segmentation (Medium)

**Goal:** Obtain a YOLOv11 segmentation model in LiteRT (`.tflite`) format.

**What is YOLOv11?** YOLO (You Only Look Once) is one of the most popular real-time object detection architectures. The v11 segmentation variant detects objects **and** generates pixel-level masks for each detected object. It can identify 80 different object categories from the COCO dataset.

**What is LiteRT?** LiteRT (formerly TensorFlow Lite) is a model format optimized for edge and mobile deployment. MATLAB can load LiteRT models using `loadLiteRTModel` and generate code for embedded targets.

This is a **medium difficulty** case: the model has **multiple output tensors** (bounding boxes, scores, and segmentation masks).

## How was this model exported?

The Ultralytics framework provides a one-line export:
```python
from ultralytics import YOLO
model = YOLO('yolo11s-seg.pt')
model.export(format='tflite')
```
This pipeline goes PyTorch → ONNX → TensorFlow SavedModel → TFLite. It requires specific TensorFlow versions, so we download the pre-exported model directly.

In [ ]:
import urllib.request
import os

# Download the pre-exported YOLOv11s-seg TFLite model from the MATLAB Deep Learning repo
url = 'https://github.com/matlab-deep-learning/pretrained-yolo-v11-litert-model-for-segmentation-and-object-detection-with-matlab/raw/main/yolo11s-seg_float32.tflite'
tflite_file = 'yolo11s-seg_float32.tflite'

if not os.path.exists(tflite_file):
    print('Downloading YOLOv11s-seg TFLite model (~39 MB)...')
    urllib.request.urlretrieve(url, tflite_file)

file_size_mb = os.path.getsize(tflite_file) / (1024 * 1024)
print(f'Downloaded: {tflite_file} ({file_size_mb:.1f} MB)')
print(f'\nThis file can be loaded in MATLAB using:')
print('  model = loadLiteRTModel("yolo11s-seg_float32.tflite")')

# Part 3: Stateful LSTM — Sensor Activity Recognition (Advanced)

**Goal:** Train a stateful LSTM on accelerometer data and export it to both `.pt2` and `.tflite` with **explicit hidden state inputs and outputs**.

## Why stateful?

In a real deployment scenario (e.g., a smartphone or microcontroller reading accelerometer data), sensor data arrives as a continuous stream. A **stateful** model carries its memory (hidden state) across inference calls:

```
Time 0: (sensor_data_0, h_init, c_init) → (prediction_0, h_1, c_1)
Time 1: (sensor_data_1, h_1,    c_1)    → (prediction_1, h_2, c_2)
Time 2: (sensor_data_2, h_2,    c_2)    → (prediction_2, h_3, c_3)
...
```

This means the model must **accept** `(input, h_0, c_0)` and **return** `(output, h_n, c_n)` — the hidden state is an explicit input/output of the model, not hidden inside it.

## The dataset

We use smartphone accelerometer data (3-axis: x, y, z) to classify human activities:
- Dancing
- Running
- Sitting
- Standing
- Walking

Each sample is 75 timesteps of 3-axis accelerometer readings.

## 3.1 Generate Synthetic Accelerometer Data

We generate synthetic data that mimics real accelerometer patterns for each activity. This keeps the tutorial self-contained (no external dataset downloads needed).

In [ ]:
import matplotlib.pyplot as plt

def generate_activity_data(num_samples_per_class=200, seq_len=75, num_features=3):
    """
    Generate synthetic 3-axis accelerometer data for 5 activities.
    Each activity has a distinct pattern in the accelerometer readings.
    """
    activities = ['Dancing', 'Running', 'Sitting', 'Standing', 'Walking']
    X_all, Y_all = [], []

    for class_idx, activity in enumerate(activities):
        for _ in range(num_samples_per_class):
            t = np.linspace(0, 2 * np.pi, seq_len)
            noise = np.random.randn(seq_len, num_features) * 0.5

            if activity == 'Sitting':
                # Low amplitude, near-constant with small noise
                data = np.stack([np.ones(seq_len) * 0.1,
                                 np.ones(seq_len) * -9.8,
                                 np.ones(seq_len) * 0.05], axis=1) + noise * 0.3
            elif activity == 'Standing':
                # Similar to sitting but with slight sway
                data = np.stack([np.sin(t * 0.5) * 0.5,
                                 np.ones(seq_len) * -9.8,
                                 np.cos(t * 0.3) * 0.3], axis=1) + noise * 0.4
            elif activity == 'Walking':
                # Periodic, moderate amplitude
                freq = np.random.uniform(1.5, 2.5)
                data = np.stack([np.sin(t * freq) * 3.0,
                                 np.sin(t * freq * 2) * 2.0 - 9.8,
                                 np.cos(t * freq) * 2.5], axis=1) + noise
            elif activity == 'Running':
                # Higher frequency, larger amplitude than walking
                freq = np.random.uniform(3.0, 4.5)
                data = np.stack([np.sin(t * freq) * 8.0,
                                 np.sin(t * freq * 2) * 6.0 - 9.8,
                                 np.cos(t * freq) * 7.0], axis=1) + noise * 2.0
            elif activity == 'Dancing':
                # Irregular, multi-frequency
                f1, f2 = np.random.uniform(1, 3), np.random.uniform(2, 5)
                data = np.stack([np.sin(t * f1) * 5.0 + np.cos(t * f2) * 3.0,
                                 np.sin(t * f2) * 4.0 - 9.8,
                                 np.cos(t * f1) * 6.0 + np.sin(t * 3) * 2.0], axis=1) + noise * 1.5

            X_all.append(data)
            Y_all.append(class_idx)

    X = np.array(X_all, dtype=np.float32)  # (N, 75, 3)
    Y = np.array(Y_all, dtype=np.int64)    # (N,)

    # Shuffle
    perm = np.random.permutation(len(X))
    return X[perm], Y[perm]

# Generate data
np.random.seed(42)
X_data, Y_data = generate_activity_data()
print(f'Dataset: {X_data.shape[0]} samples, {X_data.shape[1]} timesteps, {X_data.shape[2]} features')
print(f'Classes: Dancing(0), Running(1), Sitting(2), Standing(3), Walking(4)')

# Train/test split (80/20)
split = int(0.8 * len(X_data))
X_train, X_test = X_data[:split], X_data[split:]
Y_train, Y_test = Y_data[:split], Y_data[split:]

In [ ]:
# Visualize one sample per activity
activities = ['Dancing', 'Running', 'Sitting', 'Standing', 'Walking']
fig, axes = plt.subplots(1, 5, figsize=(18, 3), sharey=True)

for i, activity in enumerate(activities):
    idx = np.where(Y_data == i)[0][0]
    axes[i].plot(X_data[idx])
    axes[i].set_title(activity)
    axes[i].set_xlabel('Timestep')
    if i == 0:
        axes[i].set_ylabel('Acceleration')
    axes[i].legend(['X', 'Y', 'Z'], fontsize=8)

plt.suptitle('Accelerometer Patterns by Activity', fontsize=14)
plt.tight_layout()
plt.show()

## 3.2 Define the Stateful LSTM Model

The key difference from a standard LSTM: **hidden state (`h`) and cell state (`c`) are explicit inputs and outputs** of the `forward()` method. This allows the caller to manage state between inference calls.

```
Standard:  forward(x)              → predictions
Stateful:  forward(x, h_0, c_0)    → (predictions, h_n, c_n)
```

In [ ]:
class StatefulLSTM(nn.Module):
    """
    Stateful LSTM for sequence-to-sequence activity classification.

    Inputs:
        x:   (batch, seq_len, input_size)  — sensor readings
        h_0: (num_layers, batch, hidden_size) — initial hidden state
        c_0: (num_layers, batch, hidden_size) — initial cell state

    Outputs:
        out: (batch, seq_len, num_classes) — per-timestep predictions
        h_n: (num_layers, batch, hidden_size) — final hidden state
        c_n: (num_layers, batch, hidden_size) — final cell state
    """
    def __init__(self, input_size=3, hidden_size=50, num_layers=1, num_classes=5):
        super(StatefulLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, h_0, c_0):
        # LSTM processes the sequence using the provided initial states
        lstm_out, (h_n, c_n) = self.lstm(x, (h_0, c_0))

        # Apply classifier to every timestep
        out = self.fc(lstm_out)  # (batch, seq_len, num_classes)

        # Return predictions AND updated states
        return out, h_n, c_n

# Model configuration (matching the doc example)
INPUT_SIZE = 3       # 3-axis accelerometer
HIDDEN_SIZE = 50     # 50 hidden units
NUM_LAYERS = 1
NUM_CLASSES = 5      # 5 activities

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = StatefulLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, NUM_CLASSES).to(device)
print(model)
print(f'\nParameters: {sum(p.numel() for p in model.parameters()):,}')

## 3.3 Train the Stateful LSTM

During training, we initialize hidden states to zeros at the start of each sample. The model learns to update these states as it processes the sequence.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Create DataLoaders
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, Y_batch in train_loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)
        batch_size = X_batch.size(0)

        # Initialize hidden states to zeros
        h_0 = torch.zeros(NUM_LAYERS, batch_size, HIDDEN_SIZE, device=device)
        c_0 = torch.zeros(NUM_LAYERS, batch_size, HIDDEN_SIZE, device=device)

        optimizer.zero_grad()
        # Forward pass: get per-timestep predictions and final states
        out, h_n, c_n = model(X_batch, h_0, c_0)

        # Use the last timestep's prediction for classification
        last_pred = out[:, -1, :]  # (batch, num_classes)
        loss = criterion(last_pred, Y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(last_pred, 1)
        total += Y_batch.size(0)
        correct += (predicted == Y_batch).sum().item()

    if (epoch + 1) % 5 == 0:
        acc = 100.0 * correct / total
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}, Accuracy: {acc:.1f}%')

print('Training complete!')

In [ ]:
# Evaluate on test data
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for X_batch, Y_batch in test_loader:
        X_batch = X_batch.to(device)
        Y_batch = Y_batch.to(device)
        batch_size = X_batch.size(0)

        h_0 = torch.zeros(NUM_LAYERS, batch_size, HIDDEN_SIZE, device=device)
        c_0 = torch.zeros(NUM_LAYERS, batch_size, HIDDEN_SIZE, device=device)

        out, _, _ = model(X_batch, h_0, c_0)
        last_pred = out[:, -1, :]
        _, predicted = torch.max(last_pred, 1)
        total += Y_batch.size(0)
        correct += (predicted == Y_batch).sum().item()

print(f'Test Accuracy: {100.0 * correct / total:.1f}%')

## 3.4 Export the Stateful LSTM as PyTorch ExportedProgram

Notice how the export includes **three inputs** `(x, h_0, c_0)` and **three outputs** `(predictions, h_n, c_n)`. This is what makes the model "stateful" — the caller manages the hidden state.

In [ ]:
model_cpu = model.cpu()
model_cpu.eval()

# Sample inputs for export
# Note: batch=1 for single-sample inference at deployment time
sample_x = torch.randn(1, 75, 3)                          # (batch=1, seq_len=75, features=3)
sample_h0 = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)       # (num_layers=1, batch=1, hidden=50)
sample_c0 = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)       # (num_layers=1, batch=1, hidden=50)

# Export with all three inputs
exported_lstm = torch.export.export(
    model_cpu,
    (sample_x, sample_h0, sample_c0)
)

torch.export.save(exported_lstm, 'stateful_lstm.pt2')
print('Saved: stateful_lstm.pt2')
print(f'\nExported graph signature:')
print(f'  Inputs:  x {list(sample_x.shape)}, h_0 {list(sample_h0.shape)}, c_0 {list(sample_c0.shape)}')
print(f'  Outputs: predictions [1, 75, 5], h_n {list(sample_h0.shape)}, c_n {list(sample_c0.shape)}')

import litert_torch

# Convert to LiteRT with explicit state I/O
lstm_edge_model = litert_torch.convert(
    model_cpu,
    (sample_x, sample_h0, sample_c0)
)

lstm_edge_model.export('stateful_lstm.tflite')
print('Saved: stateful_lstm.tflite')
print(f'\nThe .tflite model has:')
print(f'  3 inputs:  x, h_0, c_0')
print(f'  3 outputs: predictions, h_n, c_n')
print(f'\nIn MATLAB, you will pass h_n/c_n from one invoke() call')
print(f'back as h_0/c_0 for the next call to maintain state.')

In [ ]:
import litert_torch

# Convert to LiteRT with explicit state I/O
lstm_edge_model = litert_torch.convert(
    model_cpu,
    (sample_x, sample_h0, sample_c0)
)

lstm_edge_model.export('stateful_lstm.tflite')
print('Saved: stateful_lstm.tflite')
print(f'\nThe .tflite model has:')
print(f'  3 inputs:  x, h_0, c_0')
print(f'  3 outputs: predictions, h_n, c_n')
print(f'\nIn MATLAB, you will pass h_n/c_n from one invoke() call')
print(f'back as h_0/c_0 for the next call to maintain state.')

In [ ]:
# Take one test sample and process it timestep by timestep
test_sample = torch.from_numpy(X_test[0:1])  # (1, 75, 3)
true_label = Y_test[0]

model_cpu.eval()
with torch.no_grad():
    # --- Method 1: Process entire sequence at once ---
    h_0 = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
    c_0 = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
    full_out, _, _ = model_cpu(test_sample, h_0, c_0)
    full_pred = torch.argmax(full_out[0, -1, :]).item()

    # --- Method 2: Process one timestep at a time (stateful) ---
    h = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
    c = torch.zeros(NUM_LAYERS, 1, HIDDEN_SIZE)
    for t in range(75):
        x_t = test_sample[:, t:t+1, :]  # (1, 1, 3) — single timestep
        out_t, h, c = model_cpu(x_t, h, c)  # carry state forward
    step_pred = torch.argmax(out_t[0, -1, :]).item()

activities = ['Dancing', 'Running', 'Sitting', 'Standing', 'Walking']
print(f'True activity:        {activities[true_label]}')
print(f'Full-sequence pred:   {activities[full_pred]}')
print(f'Step-by-step pred:    {activities[step_pred]}')
print(f'\nBoth methods agree: {full_pred == step_pred}')

---
## Step 4: Download All Exported Models

Download all model files to your local machine for use in the MATLAB/Simulink portion of the tutorial.

In [ ]:
from google.colab import files
import os

export_files = [
    'repvit.pt2',
    'yolo11s-seg_float32.tflite',
    'stateful_lstm.pt2',
    'stateful_lstm.tflite'
]

for f in export_files:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f'Downloading {f} ({size_mb:.1f} MB)...')
        files.download(f)
    else:
        print(f'Warning: {f} not found — check export cells above')

print('\nDone! You now have all models ready for MATLAB and Simulink.')

---
## Summary

| Model | Format | File | MATLAB Function | Difficulty |
|-------|--------|------|----------------|------------|
| RepViT (classifier) | PyTorch `.pt2` | `repvit.pt2` | `loadPyTorchExportedProgram` | Simple |
| YOLOv11 (detector + segmentor) | LiteRT `.tflite` | `yolo11s-seg_float32.tflite` | `loadLiteRTModel` | Medium |
| Stateful LSTM (activity recognition) | PyTorch `.pt2` | `stateful_lstm.pt2` | `loadPyTorchExportedProgram` | Advanced |
| Stateful LSTM (activity recognition) | LiteRT `.tflite` | `stateful_lstm.tflite` | `loadLiteRTModel` | Advanced |

### Key takeaways

1. **Simple models** — `torch.export.export()` + `torch.export.save()` is all you need
2. **Multi-output models** — frameworks like Ultralytics handle the conversion pipeline
3. **Stateful models** — make hidden state explicit as model inputs/outputs; this works for both `.pt2` and `.tflite`

### What's next? (Live Demo)

In the MATLAB/Simulink live demo, we will:
1. Load `repvit.pt2` with `loadPyTorchExportedProgram` and generate C/C++ code
2. Load `yolo11s-seg_float32.tflite` with `loadLiteRTModel` and run object detection
3. Load `stateful_lstm.tflite` with explicit state management in Simulink for real-time activity classification